# Day 3.6 — Tools with Side Effects

## Before you begin

### Learning outcomes

- Separate reads, reversible writes, external actions and destructive actions.
- Watch a direct function call change state with nothing checking it first.
- See the same call refused once it goes through the agent.

Architecture reference: [Day 3 diagrams D11](../../diagrams/source/day_03.md).

### Expected observation

Calling `view_calendar` leaves the workspace identical. Calling `delete_all_tasks` directly empties the task list. Routing the same call through the agent returns `denied` and the tasks survive.

## Concept briefing

## Tools that change the world

Reading and writing are not the same risk. A read can be repeated, cached and undone by
doing nothing. A write changes state, and some writes leave the machine entirely: an
email that has been sent cannot be recalled by deleting a row.

So tools are classified before they are exposed: read-only, reversible local write,
external action, destructive action. The classification is the input to policy, and it is
written by the engineer, not proposed by the model. A tool's description is documentation
that helps a model choose; it is never a security boundary, because calling a function
directly bypasses every word of it.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/safe_task_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

## Step 1 — A simulated workspace

No real calendar and no real email account are connected. `SimulatedWorkspace` is a plain
dataclass holding four lists, so every side effect is something you can print.

In [ ]:
from safe_task_agent.tools import POLICY, SimulatedWorkspace, tool_registry

workspace = SimulatedWorkspace()
tools = tool_registry(workspace)

def show(label):
    print(f"{label:<26} calendar={len(workspace.calendar)} tasks={len(workspace.tasks)} "
          f"drafts={len(workspace.drafts)} sent={len(workspace.sent)}")

print("Tools available:", list(tools))
show("starting state:")

## Step 2 — A read-only tool changes nothing

Run it twice. The state is identical, which is what makes reads cheap and safe to retry.

In [ ]:
show("before view_calendar:")
print("returned:", tools["view_calendar"]())
tools["view_calendar"]()                      # running it again is harmless
show("after two reads:")

## Step 3 — A reversible local write

Creating a draft changes our own state, but nothing has left the machine: we could delete the
draft and be back where we started.

In [ ]:
show("before create_draft:")
draft = tools["create_draft"](to="mentor@example.test", subject="Update", body="Synthetic only")
print("returned:", draft)
show("after create_draft:")

## Step 4 — An external action leaves the machine

In this course `send_email` only appends to a list, but it stands for the real thing: once a
message is delivered you cannot un-deliver it.

In [ ]:
show("before send_email:")
tools["send_email"](**draft)                  # called directly - nothing asked permission
show("after send_email:")
print("\nNo policy check ran. We called the Python function ourselves.")

## Step 5 — A destructive action, called directly

Watch the task list disappear. Note what did *not* happen: no check, no approval, no record.

In [ ]:
print("tasks before:", workspace.tasks)
removed = tools["delete_all_tasks"]()
print("deleted     :", removed, "tasks")
print("tasks after :", workspace.tasks)
print("\nA tool description saying 'dangerous, ask first' would not have stopped this.")
print("Descriptions guide a model's choice; they are not a security boundary.")

## Step 6 — The same request through the agent

Now go through `SafeTaskAgent`, which consults the policy table before touching a tool.

In [ ]:
from safe_task_agent import ActionRequest, SafeTaskAgent

agent = SafeTaskAgent()                       # a fresh workspace of its own
print("tasks before:", agent.workspace.tasks)

result = agent.request(ActionRequest("delete_all_tasks", {}, reason="tidy up"))
print("status      :", result.status)
print("message     :", result.message)
print("tasks after :", agent.workspace.tasks)
print("\nSame function, same arguments. The difference is who was allowed to call it.")

### Try it yourself

Predict the decision for each of the four tools before running. Which one pauses rather than
completing or being refused?

In [ ]:
# --- Worked solution ---
requests = [
    ActionRequest("view_calendar", {}),
    ActionRequest("create_draft", {"to": "m@example.test", "subject": "S", "body": "Synthetic"}),
    ActionRequest("send_email", {"to": "m@example.test", "subject": "S", "body": "Synthetic"}),
    ActionRequest("delete_all_tasks", {}),
]
fresh = SafeTaskAgent()
for request in requests:
    outcome = fresh.request(request)
    print(f"{request.tool:<18} policy={POLICY.get(request.tool, 'deny'):<9} status={outcome.status}")
print("\nemails actually sent:", len(fresh.workspace.sent))

# send_email is the one that PAUSES: policy says "approval", so the agent stores the
# request and waits for a human. Nothing was sent. Day 3.7 completes that handshake.

### Checkpoint

**1. Why treat `create_draft` and `send_email` differently when both write data?**

<details><summary>Show answer</summary>

A draft is a reversible local write: we can delete it and no one outside ever saw it. Sending is an external action that leaves the machine and cannot be recalled. The question is not 'does it write?' but 'can it be undone?'.

</details>

**2. A tool's description says it must never be used without permission. Is that a guardrail?**

<details><summary>Show answer</summary>

Not a load-bearing one. The description is documentation for whoever chooses the tool; Step 5 changed state with a direct call and no description was consulted. Enforcement has to sit in the host code that decides whether the function is called at all.

</details>

### Recap

- **Limitation we saw:** A direct call to `delete_all_tasks` wiped the task list with no check and no record.
- **Layer we added:** A tool registry with an explicit risk class per tool, reached only through the agent.
- **Evidence it worked:** The same destructive request returned `denied` through the agent and the tasks were still there afterwards.